# Introduction to Audio Machine Learning
## Week 9 Worksheet - Feature Extraction and Convolutional Layers
-----
Welcome! In this notebook, we’ll look at feature extraction and saving extracted features in Python, as well as defining and training a convolutional neural network to an audio classification problem.

There are weekly computer lab sessions, 12:10-13:00 , or 13:10-14:00, on Wednesdays in Computer Lab B.51, Hugh Robson Building. To get the most out of these sessions, I **strongly** recommend you work through the worksheet in your own time before attending the session.

In [ ]:
### Setup!
%pip install soundfile #This downloads and installs 'soundfile'
%pip install librosa   #This downloads and installs 'librosa'
import librosa
import soundfile as sf
import IPython
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy
from glob import glob
from os.path import join
import sklearn
import torch

---
# 1 - Dataset Exploration

The dataset for this worksheet is included in the GitHub repository, in a directory called `Multiclass_Inst`.

Explore the dataset and see what it contains.

---

---
### ✏️✏️ Exercise 1.1: Get a list of filenames ✏️✏️
---

The dataset holds all audio examples in a single directory. Use glob to find the filenames for every file in the directory.



In [ ]:
filenames = # Your code here

In [ ]:
assert len(filenames) == 1330, 'There should be 1330 .wav files in filenames'

This gives you a list of filenames, which describes the location of each audio file in your dataset.

---
### ✏️✏️ Exercise 1.2: Extract label from filename ✏️✏️
---

Create a function that takes a filename string, such as:

string = `'Multiclass_Inst/clarinet_10.wav'`

as argument. The function should return the instrument class name.

*Hint*: Use the string methods, `.split()` or `.rsplit()`, and `os.path.sep`.

In [ ]:
def extract_label(filename):
    return #instrument_class_name

In [ ]:
# Test your function here
assert extract_label('Multiclass_Inst/clarinet_10.wav') == 'clarinet'
assert extract_label('Multiclass_Inst/flute_140.wav') == 'flute'

----

Now you should have function that takes a filepath, and returns the instrument class.

`extract_label('Multiclass_Inst/trumpet_210.wav')`

returns

`'trumpet'`

---
### ✏️✏️ Exercise 1.3: Count Class occurrences ✏️✏️
---

Use your `extract_label()`, the `filenames` list, and a for-loop, to count how many occurrences of each class are in your dataset.

----

You can load an audio file, and find its length in seconds, as follows:

In [ ]:
x, sr = sf.read(filenames[0])

len_in_s = x.shape[0]/sr

print(f'audio file: {filenames[0]}, \n sample rate: {sr}, \n length in seconds: {len_in_s}')

---
### ✏️✏️ Exercise 1.4: Find audio file lengths ✏️✏️
---

Now, for each class, find the lengths of audio files in the dataset. Create a separate list for each of the three classes in the dataset. For each class, the list should hold a number representing the length in seconds for each audio file. So, if there are 50 clarinet files, the list for clarinet should hold 50 numbers, representing the lengths of each file.

You can create a histogram of the data, to visualise how the lengths of audio file vary within each class. An example of creating a histogram is shown below:

In [ ]:
data = np.random.randn(500) + 5
plt.hist(data)
plt.xlabel('Length in Seconds')
plt.ylabel('Number of occurrences')

---
### ✏️✏️ Exercise 1.5: Visualise Audio Lengths ✏️✏️
---

For each instrument class, do the following:
- Convert the list of audio file lengths into a numpy array
- Find and print the minimum, maximum, and mean of audio file lengths for that class
- Plot a histrogram showing the distribution of file lengths for each class

In [ ]:
# Your code here

---
# 2 - Feature Extraction

The following code will extract some features from your audio dataset. I will demonstrate two approaches to feature extraction. Firstly, we will extract some features such as spectral centroid, using librosa. In the next section, we will use Convolutional Neural Networks.

---
## 2.1 - Time-Averaged Features

When using a non-sequence model, such as logistic regression, we need to work with a fixed number of features. In audio data, we can handle this by averaging audio features over time.

In [ ]:
# Load the audio file
x, sr = sf.read(filenames[0])

# Extract Spectral Centroid
spec_cent = librosa.feature.spectral_centroid(y=x, sr=sr)

The above code extracts the spectral centroid of the audio file, over time:

In [ ]:
print(spec_cent.shape)
print(f'One feature, over {spec_cent.shape[1]} time steps')

We can time-average these features, to create a fixed length number of features for each audio file. We can use the mean and standard deviation of each feature.

In [ ]:
spec_cent_mean = np.mean(spec_cent)
spec_cent_std = np.std(spec_cent)

## 2.2 - Feature Extraction Function

The below function extracts a range of features from an audio file, using Librosa.

In [ ]:
def extract_features(x, sr):
    x = np.asarray(x, dtype=np.float32)
    # Remove silence from beginning and end of audio
    x, _ = librosa.effects.trim(x)

    # Extact each feature using librosa
    centroid = librosa.feature.spectral_centroid(y=x, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=x, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=x, sr=sr, roll_percent=0.85)
    flatness = librosa.feature.spectral_flatness(y=x)
    contrast = librosa.feature.spectral_contrast(y=x, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y=x)
    rms = librosa.feature.rms(y=x)

    # Time-Average features
    features = [
        np.mean(centroid), np.std(centroid),
        np.mean(bandwidth), np.std(bandwidth),
        np.mean(rolloff), np.std(rolloff),
        np.mean(flatness), np.std(flatness),        
        np.mean(contrast), np.std(contrast),
        np.mean(zcr), np.std(zcr),
        np.mean(rms), np.std(rms),        
    ]

    # Return features
    return np.stack(features)

The function usage is shown below, and the result can be seen, with 14 features being extracted in total.

In [ ]:
x, sr = sf.read(filenames[0])
feats = extract_features(x, sr)

## 2.3 - Extracting Features over the Dataset

The following code will extract features for every dataset item, saving the result in a numpy array.

In [ ]:
from tqdm import tqdm
feature_list = []
label_list = []
file_id_list = []

# tqdm allows you watch the progress of a for-loop
for n in tqdm(filenames):
    # Load audio file
    x, sr = sf.read(n)
    # Trim each file to max 5 seconds - to reduce computation time
    x = x[0:5*sr]
    # Save features to feature_list
    feature_list.append(extract_features(x, sr))

    lab = n.rsplit(os.path.sep, 1)[-1]
    file_id_list.append(lab.split('_')[1].split('.')[0])
    lab = lab.split('_')[0]
    
    label_list.append(lab)

X = np.stack(feature_list)

The resulting feature array, X, contains 14 features, for each item in the dataset:

In [ ]:
print(f'X contains {X.shape[0]} data examples, and {X.shape[1]} features')

To save our features, we can create a pandas dataframe to hold all the information. The below cell creates a dataframe with a column for each feature.

In [ ]:
import pandas as pd

feature_names = [
    'mean_centroid', 'std_centroid',
    'mean_bandwidth', 'std_bandwidth',
    'mean_rolloff', 'std_rolloff',
    'mean_flatness', 'std_flatness',
    'mean_contrast', 'std_contrast',
    'mean_zcr', 'std_zcr',
    'mean_rms', 'std_rms',
]

df = pd.DataFrame(X, columns = feature_names)

In [ ]:
# View the first 5 rows of the dataframe
df.head()

Now we can add the class labels:

In [ ]:
df.insert(0, 'Class', label_list)

In [ ]:
# View the first 5 rows of the dataframe
df.head()

Now we can add the file_ids (So we can see exactly which file each row was extracted from):

In [ ]:
df.insert(1, 'file_id', file_id_list)

In [ ]:
df.head()

Now, saving the .csv file:

In [ ]:
df.to_csv('time_av_features.csv')

---
### ✏️✏️ Exercise 2.1: Fit a Logistic Regression Model ✏️✏️
---

Use the features, `X` and labels, `label_list`, to fit a Logistic Regression model using `sklearn`. You should do the following:

- Split the data using `sklearn.model_selection.train_test_split()`
- Use an `sklearn.preprocessing.StandardScaler()` to scale the training dataset
- Fit an `sklearn.linear_model.LogisticRegression()` model on the training features and labels
- Use the resulting models to make predictions on the test dataset. (Be sure to scale your test features using the same scaler used to scale your training features!)
- Evaluate the accuracy of the classifier. Find the overall accuracy, and precision and recall for each class. You can use the `sklearn.metrics.classification_report()` for this.

In [ ]:
# Your code here

# 3 - 1D Convolutional Layers

In this section we will introduce Convolutional Layers, the building blocks of Convolutional Neural Networks.

In [ ]:
import torch

## 3.1 -  Using a 1D Convolutional layer

We can create a 1D-Convolutional layer as follows:

In [ ]:
layer = torch.nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5)

Now we will create some audio to process with the layer, you can process the audio using the layer.forward() function.

In [ ]:
input_audio = torch.randn(5, 1, 44100)
print(f'input shape: {input_audio.shape}')
output = layer(input_audio)
print(f'output shape: {output.shape}')

Note the dimensions of the input and output of the convolutional layer. The dimensions of an input to a 1D-Convolutional layer have the following meaning:

$[N, C, T]$

- $N$ is the number of data examples being input to the layer
- $C$ is the number of channels
- $T$ is the length of the sequence

For a convolutional layer, the number of channels of input is defined when the layer is created, by setting the argument `in_channels`. The number of channels that will be in the layer's output is set by the argument `out_channels`. An input given to a convolutional layer must have the same number of channels as the `in_channels` defined when the layer was created.

The below code attemps to process audio with 10 channels using the convolutional layer from earlier:

In [ ]:
wrong_channels_audio = torch.randn(5, 10, 44100) 
layer(wrong_channels_audio)

Read the above error message. This one is quite common! If you get this, you know you have a channel mismatch problem.

---
### ✏️✏️ Exercise 3.1: 1D-Convolutional Processing ✏️✏️
---

For the audio defined in the following cell:

In [ ]:
x = torch.randn(10, 4, 2000)

Create a `torch.nn.conv1d` layer that can successfully process this input. Set the `in_channels` appropriately according to the input `x`. The output of the layer should have 16 channels. Set the `kernel_size` to 5.

In [ ]:
conv_layer = torch.nn.Conv1d(# Your code here)

In [ ]:
# Check your answer
try:
    conv_layer(x)
    assert(list(conv_layer(x).shape) == [10, 16 ,1996]), 'output is incorrect shape'
except RuntimeError:
    print('Wrong number of input channels in conv_layer')

---
### ✏️✏️ Exercise 3.2: More 1D-Convolutional Processing ✏️✏️
---

Now create two `torch.nn.conv1d` layers. The input is defined as:

In [ ]:
x = torch.randn(10, 8, 2000)

The output of the first layer should have 16 channels. The output of the second layer should have 1 channel. Use a kernel size of 5 for each layer.

In [ ]:
layer_1 = torch.nn.Conv1d(# Your code here)
layer_2 = torch.nn.Conv1d(# Your code here)
# Your code here

In [ ]:
# Check your answer
try:
    layer_2(layer_1(x))
    assert(list(layer_2(layer_1(x)).shape) == [10, 1, 1992]), 'output is incorrect shape'
except RuntimeError:
    print('Channel mismatch error')

# 4 -  Processing spectrograms with 2D Convolutional Layers

We’ll move from 1D convolutions on waveforms to 2D convolutions on spectrograms. In PyTorch, spectrograms are represented as tensors with the shape:

$[N, C, F, T]$

- $N$ is the number of data examples being input to the layer
- $C$ is the number of channels
- $F$ is the number of frequency bins in the spectrogram (the y-axis of the image)
- $T$ is the length of time frames in the spectrogram (the x-axis of the image)

A single spectrogram in PyTorch should have the following dimensions:

$[1, 1, F, T]$

It is important when inputting spectrograms to 2d Convolutional layers that there are 4 dimensions, even if the number of data examples and number of channels are both equal to 1.

## 4.1 - Setup a sample spectrogram tensor

In [ ]:
x = np.random.randn(1,44100) # Create a five random audio examples

In [ ]:
# Create spectrogram of audio in librosa
S = librosa.stft(x, n_fft=2048, hop_length=512, window="hann")
Spect = librosa.amplitude_to_db(np.abs(S), ref=np.max)

We can visualise this spectrogram.

In [ ]:
plt.figure(figsize=(6, 3))
plt.imshow(Spect[0,:,:], origin="lower", aspect="auto", cmap="magma")
plt.colorbar(label="dB")
plt.title(f"White Noise")
plt.xlabel("Time frames")
plt.ylabel("Frequency bins")
plt.tight_layout()
plt.show()

To process this spectrogram using PyTorch, we must convert it from a numpy array into a PyTorch tensor:

In [ ]:
# Convert spectrogram to torch, from numpy array
x = torch.from_numpy(Spect.astype(np.float32))

In [ ]:
print("Input tensor:", x.shape)

Note that PyTorch 2d convolutional layers expect 4 dimensions, so we can add an extra dimensions as follows:

In [ ]:
x = x.unsqueeze(1)
print("Input tensor:", x.shape)

## 3.2 - Using a 2D Convolutional Layer

The 2D layer works by sliding a filter kernel over 2 dimensions. The below code defines a 2d convolutional layer and applies it to the spectrogram.

In [ ]:
conv2d = torch.nn.Conv2d(in_channels=1, out_channels=8, kernel_size=(3, 3))
y1 = conv2d(x)
print("Input dimensions:", x.shape)  # (1, 8, F, T)
print("Conv 3x3, no padding, output dimensions:", y1.shape)  # (1, 8, F, T)

The filter kernel can be adjusted to change how the filter is applied, which in turn affects the shape of the output:

In [ ]:
# Frequency-only pattern: tall kernel, time=1
conv_freq = torch.nn.Conv2d(1, 8, kernel_size=(5, 1))  # preserves F,T
y2 = conv_freq(x)
print("Conv 5x1 (freq-only):", y2.shape)

# Time-only pattern: wide kernel, freq=1
conv_time = torch.nn.Conv2d(1, 8, kernel_size=(1, 7))
y3 = conv_time(x)
print("Conv 1x7 (time-only):", y3.shape)

# Downsampling time with stride 2 
conv_time_stride = torch.nn.Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2))
y4 = conv_time_stride(x)
print("Conv 3x3, stride=2:", y4.shape)

For more details on how 2D convolutional layers work, look at the appropriate section on this website:

https://poloclub.github.io/cnn-explainer/

## 3.3 - Pooling Layers

Pooling layers can be introduced to reduce resolution. Here are some common PyTorch pooling layers:

`torch.nn.MaxPool2d`: picks maximum in each window

`torch.nn.AvgPool2d`: averages in each window


In [ ]:
pool2 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
x = torch.arange(0, 36).reshape(1,1,6,6)
print(f'input to max pool layer: \n{x}')
print(f'output of max pool layer: \n{pool2(x)}')

In [ ]:
avg2 = torch.nn.AvgPool2d(kernel_size=2, stride=2)
x = torch.arange(0, 36, dtype=torch.float).reshape(1,1,6,6)
print(f'input to average pool layer: \n{x}')
print(f'output of average pool layer: \n{avg2(x)}')

Pooling can also be applied across one-dimension only. For example, to pool in the time-dimension of a spectrogram only:

In [ ]:
avg2 = torch.nn.AvgPool2d(kernel_size=2, stride=(1,2))
x = torch.arange(0, 36, dtype=torch.float).reshape(1,1,6,6)
print(f'input to average pool layer: \n{x}')
print(f'output of average pool layer: \n{avg2(x)}')

## 3.4 - Nonlinear activations

Convolutional layers are usually followed by nonlinearities.

Common nonlinear activation functions used in Convolutional Neural Networks are:

- ReLU
- LeakyReLU

These are defined in PyTorch below:

In [ ]:
relu = torch.nn.ReLU()
lrelu = torch.nn.LeakyReLU(negative_slope=0.1)

In [ ]:
z = torch.linspace(-5, 5, 100)
plt.plot(z, relu(z))
plt.title('ReLU activation function')
plt.xlabel('input')
plt.ylabel('output')
plt.grid()

In [ ]:
plt.plot(z, lrelu(z))
plt.title('Leaky ReLU activation function')
plt.xlabel('input')
plt.ylabel('output')
plt.grid()

## 3.5 - A 2D Convolution Block

Convolutional Neural Networks are usually built out of repeated Convolution Blocks. Each block applies some convolutional layer, followed by activation layer and pooling. Here is an example of a block:

In [ ]:
class ConvBlock2D(torch.nn.Module):
    def __init__(self, c_in, c_out, k=(3,3), pool=2, act="relu"):
        super().__init__()
        self.conv = torch.nn.Conv2d(c_in, c_out, kernel_size=k)
        if act == "relu":
            self.act = torch.nn.ReLU()
        elif act == "lrelu":
            self.act = torch.nn.LeakyReLU(0.1, inplace=True)
        self.pool = torch.nn.AvgPool2d(kernel_size=pool, stride=pool)

    def forward(self, x):
        x = self.conv(x)
        x = self.act(x)
        x = self.pool(x)
        return x

# Create a random input
x = torch.randn(1, 1, 500, 500)
block = ConvBlock2D(1, 16, k=(5,5), pool=2)
print("Block input shape:", x.shape)
out = block(x)
print("Block output shape:", out.shape)

You can create a larger network by stacking several blocks, and passing inputs through each of them in turn.

---
### ✏️✏️ Exercise 4.1: Shape sanity check✏️✏️
---

Create a Conv2d layer with kernel_size=(7,3) and stride=(1,2). Pass x and report the output shape. Which dimension changed and by what factor?
How does increasing the kernel size influence the shape of the output?
How does changing the stride influence the shape of the output?

In [ ]:
x = torch.randn(1, 1, 500, 500)
layer2d = #Your code here

---
### ✏️✏️ Exercise 4.2: Time-only vs freq-only pooling✏️✏️
---

Apply a MaxPool2d layer with kernel=(1,8), and a AvgPool2d layer with kernel=(8,1) to x. Verify which dimension is reduced each time.

In [ ]:
x = torch.randn(1, 1, 500, 500)
maxpool = # Your code here
avgpool = # Your code here

# 🎯🎯 Challenge! 🎯🎯

Use the `ConvBlock2D` class from earlier, to create a new neural network class, `ConvNet`. The new class should create a number of `ConvBlock2D` layers. The forward function should process the input to the network through each `ConvBlock2D` layer.

In [ ]:
class ConvNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Your Code Here

    def forward(self, x):
        #Your Code Here

After defining the class, create an instance of the ConvNet, and process some example spectrograms with it.

In [ ]:
Model = ConvNet()# Your code here